# Processing HIST_UVnudge data

### Set up
#### Packages

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
from scipy import stats
import warnings
warnings.simplefilter('ignore', UserWarning)
warnings.filterwarnings('ignore')
import datetime as dt
from datetime import timedelta
from Processing_functions import FixLongitude, FixTime, FixGrid, InterPlevels
xr.set_options(display_expand_data=False);
xr.set_options(keep_attrs=True);
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client
from dask.diagnostics import ProgressBar
from xgcm import Grid
from collections import defaultdict

#### Filepaths & name variables

In [2]:
## Test numbers
tst_nums = np.arange(1,4)

## File name
run_name = 'HIST_UVnudge_LM'
name_s = 'b.e21.'
name_e ='.f09_g17.'+run_name

## Filepaths
path_to_arch = "/glade/campaign/univ/ucub0155/glydia/"
comp = 'atm'
freq = 0 # 0: monthly, 1: daily
var_ind = 7

# ATM
# 0,  7,   8, 9, 11,     15,     16, 17,     18,       19,      20,    21     
# TS, PSL, U, V, TREFHT, RESTOM, Z3, FSNTOA, TGCLDLWP, AEROD_v, PRECS, PREC

# ICE
# 0,    1,  2,     3,     4,      5
# aice, hi, meltt, meltb, congel, frazil

# OCN
# 0
# MOC

# Variables
var_list = {'atm': ['TS','FLDS','CLOUD','FLNS','FSNS','FLNT','FSNT','PSL','U','V','T','TREFHT',
                    'Target_U','Target_V','Target_T','RESTOM','Z3','FSNTOA','TGCLDLWP','AEROD_v',
                    'PRECS','PRECR'],
            'ice': ['aice','hi','meltt','meltb','congel','frazil'],
            'ocn': ['MOC','TEMP']}
var_ext = {0: '', 1: '_d'}
var = var_list[comp][var_ind]+var_ext[freq]

var_eq = {'PRECS': ['PRECSC',1,'PRECSL'],
          'PRECR': ['PRECC',1,'PRECL',0,'PRECSC',0,'PRECSL']}

# Extensions
h_ext = {'atm': ['.h0.'],
       'ice': ['.h.','.h1.'],
       'ocn': ['.h.']}
mod_com = {'atm': 'cam',
           'ice': 'cice',
           'ocn': 'pop'}
time_path = {'atm': ['month_1'],
                'ice': ['month_1','day_1'],
                'ocn': ['month_1']}
compset = {0: 'BHISTsmbb',
         1: 'BSSP370smbb'}
yr_extn = {'in': [[".195001-201412.",".201501-202312.",".202401-202412."], ".*0101-*1231."],
           'out': [".195001-202412.", ".19500101-20241231."]}
vert_lev = {'atm': [False,False,True,False,False,False,False,False,True,True,True,
                    False,True,True,True,False,True,False,False,False,
                    False,False,False,False],
            'ice': [False,False,False,False,False,False],
            'ocn': [False,False]}

path_to_outdata = '/glade/work/glydia/Arctic_controls_processed_data/processed_'+run_name+'_data/'

In [3]:
cluster = PBSCluster(cores    = 1,
                     memory   = '25GiB',
                     queue    = 'casper',
                     walltime = '02:00:00',
                     account  = 'UCUB0137',
                     log_directory = '/glade/u/home/glydia/python/logs/',
                     name='HISTSSP370smbb_'+var)
cluster.scale(4*9)
client = Client(cluster)

In [4]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/glydia/Arctic_breakdown/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/glydia/Arctic_breakdown/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.117:42641,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/glydia/Arctic_breakdown/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [5]:
## Chunking variables
time_ch = 365*2 if freq == 1 else 600
chunks = {
    'atm': {'time': time_ch, 'lat': 96, 'lon': 144, 'lev': -1},
    'ice': {'time': time_ch, 'nj': 192, 'ni': 160},
    'ocn': {'time': time_ch, 'nlat': 64, 'nlon': 96, 'z_t': 5}
}

### Load & modify data
#### Control data

In [6]:
def buildFileList(v):
    ds_name_list = []
    for k in range(0,3):
        ki = 0 if k < 1 else 1
        tst_name = name_s+compset[ki]+name_e+'.'+str(i).zfill(3)
        filepath = path_to_arch+tst_name+'/'+comp+'/proc/tseries/'+time_path[comp][freq]+'/'
        if k < 2:
            filename = tst_name+h_ext[comp][freq]+v+yr_extn['in'][freq][k]+'nc'
        else:
            filename = tst_name+'.'+mod_com[comp]+h_ext[comp][freq]+v+yr_extn['in'][freq][k]+'nc'
        print(filename)
        ds_name_list.append(filepath+filename)
    return ds_name_list

In [7]:
%%time

yr_range = np.array([str(i).zfill(4) for i in np.arange(1950,2026)])
horiz_only = not vert_lev[comp][var_ind]
if var in var_eq:
    multvar = True
    vareq = var_eq[var]
    varsub = vareq[::2]
    varlen = len(vareq)
else:
    multvar = False

ens_list = []
for i in tst_nums:
    out_name = name_s+compset[0]+name_e
    
    ## Load data
    # Open dataset
    if multvar:
        ds_name_list = buildFileList(varsub[0])
        dsv = xr.open_mfdataset(paths=ds_name_list,chunks=chunks[comp],parallel=True)
        dsv = dsv[varsub[0]]
        sumstring = varsub[0]
        
        for p in np.arange(1,varlen-1,2):
            ds1_name_list = buildFileList(vareq[p+1])
            ds1 = xr.open_mfdataset(paths=ds1_name_list,chunks=chunks[comp],parallel=True)
            # Add
            if vareq[p]:
                sumstring += ' + '+vareq[p+1]
                dsv = dsv + ds1
            # Subtract
            else:
                sumstring += ' - '+vareq[p+1]
                dsv = dsv - ds1
            dsv = dsv[vareq[p+1]]

        dsv = dsv.rename(var)
        print(var+' = '+sumstring)
    
    else:
        ds_name_list = buildFileList(var)
        ds = xr.open_mfdataset(paths=ds_name_list,chunks=chunks[comp],parallel=True)
        dsv = ds[var] if horiz_only else ds

    processed_list = []
    for j in range(0,len(yr_range)-1):
        # If monthly data
        if freq == 0:
            startyr = yr_range[j]
            endyr = yr_range[j+1]
            ann_slice = dsv.sel(time=slice(startyr+'-02-01',endyr+'-01-17')) #if add_cyclic else dsv.sel(time=slice(startyr+'-02-10',endyr+'-01-17'),lat=slice(0,90))
            print('sliced '+startyr+'-02-01 to '+endyr+'-01-17')
            
            fixedtime_data = dask.delayed(FixTime)(ann_slice)
            print('   fixed time')
    
            if comp == 'ice':
                fixedgrid_data = dask.delayed(FixGrid)(fixedtime_data,'ice','gx1v7')
                processed_list.append(fixedgrid_data)
                print('   fixed CICE grid')
    
            elif comp == 'ocn':
                if var == 'TEMP':
                    fixedtime_data = fixedtime_data.sel(z_t=0,method='nearest')
                    fixedgrid_data = dask.delayed(FixGrid)(fixedtime_data,'ocn','gx1v7')
                    processed_list.append(fixedgrid_data)
                    print('   fixed CICE grid')
                else:
                    processed_list.append(fixedtime_data)
    
            else:
                fixedgrid_data = dask.delayed(FixLongitude)(fixedtime_data, False)
                # If 3D data, interpolate to pressure levels
                if vert_lev[comp][var_ind]:
                    addplev_data = dask.delayed(InterPlevels)(fixedgrid_data, var)
                    processed_list.append(addplev_data)
                else:
                    processed_list.append(fixedgrid_data)
                print('   fixed longitude')
    
                
    if horiz_only and freq == 0:
    
        processed_comp = dask.compute(*processed_list)
        print('computed list for ensemble member '+str(i).zfill(3))
        
        processed_out = xr.concat(processed_comp,dim='time').chunk({'time':111})
        ens_list.append(processed_out)
        print('concatenated data for ensemble member '+str(i).zfill(3))
    else:
        ens_list.append(processed_list)
        print('add uncomputed list for '+str(i).zfill(3))

b.e21.BHISTsmbb.f09_g17.HIST_UVnudge_LM.001.h0.PSL.195001-201412.nc
b.e21.BSSP370smbb.f09_g17.HIST_UVnudge_LM.001.h0.PSL.201501-202312.nc
b.e21.BSSP370smbb.f09_g17.HIST_UVnudge_LM.001.cam.h0.PSL.202401-202412.nc
sliced 1950-02-01 to 1951-01-17
   fixed time
   fixed longitude
sliced 1951-02-01 to 1952-01-17
   fixed time
   fixed longitude
sliced 1952-02-01 to 1953-01-17
   fixed time
   fixed longitude
sliced 1953-02-01 to 1954-01-17
   fixed time
   fixed longitude
sliced 1954-02-01 to 1955-01-17
   fixed time
   fixed longitude
sliced 1955-02-01 to 1956-01-17
   fixed time
   fixed longitude
sliced 1956-02-01 to 1957-01-17
   fixed time
   fixed longitude
sliced 1957-02-01 to 1958-01-17
   fixed time
   fixed longitude
sliced 1958-02-01 to 1959-01-17
   fixed time
   fixed longitude
sliced 1959-02-01 to 1960-01-17
   fixed time
   fixed longitude
sliced 1960-02-01 to 1961-01-17
   fixed time
   fixed longitude
sliced 1961-02-01 to 1962-01-17
   fixed time
   fixed longitude
sliced 1

In [8]:
%%time

if not vert_lev[comp][var_ind] and freq == 0:
    ens_comp = dask.compute(*ens_list)
    print('computed list of ensemble members')
    
    ens_out = xr.concat(ens_comp,pd.Index(tst_nums,name='ensemble_member'))
    print('concatenated data of ensemble members')
    
    ens_out.to_netcdf(path_to_outdata+out_name+h_ext[comp][freq]+var+yr_extn['out'][freq]+'nc', 
                                format='NETCDF4',encoding={var: {"zlib": True, "complevel": 1}})
    print('wrote data to disk')
    
else:
    ens_comp = dask.compute(*[i[0] for i in ens_list])
    ens_out = xr.concat(ens_comp,pd.Index(tst_nums,name='ensemble_member'))
    ens_out.to_zarr(path_to_outdata+out_name+h_ext[comp][freq]+var+yr_extn['out'][freq]+'zarr', 
                            group=var)
    print('saved initial zarr store')
    for i in range(1,len(yr_range)-1):
        yr = str(yr_range[i])
        print('   saving year '+yr)
        
        ens_comp = dask.compute(*[j[i] for j in ens_list])
        ens_out = xr.concat(ens_comp,pd.Index(tst_nums,name='ensemble_member'))
        
        ens_out.sel(time=slice(yr+'-01-01',yr+'-12-31')).to_zarr(path_to_outdata+out_name+h_ext[comp][freq]+var+yr_extn['out'][freq]+'zarr', 
                            append_dim='time', mode='a-',group=var)
    print('wrote data to disk')

computed list of ensemble members
concatenated data of ensemble members


HDF5-DIAG: Error detected in HDF5 (1.14.6) MPI-process 0:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/glade/work/gl

wrote data to disk
CPU times: user 9.75 s, sys: 1.31 s, total: 11.1 s
Wall time: 11.1 s


In [9]:
client.shutdown()